## Cell 1: Setup
**What this demonstrates**: Environment initialisation for simplified FLARE — one Anthropic model (Haiku) for iterative sentence generation, OpenAI embeddings + Chroma for retrieval when uncertainty is detected. This is a simplified implementation: full FLARE uses per-token log-probabilities to detect uncertainty; this notebook detects hedging language instead, which is practical with any LLM API.
**Expected output**: Setup confirmation with model name, hedge word list, and MAX_SENTENCES limit.

In [ ]:
%pip install -q langchain langchain-community langchain-openai chromadb anthropic openai python-dotenv

import os
import re
import time
import pathlib
from dataclasses import dataclass, field

from dotenv import load_dotenv, find_dotenv
from anthropic import Anthropic
from langchain_community.vectorstores import Chroma
from langchain_core.documents import Document
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

load_dotenv(find_dotenv())
_anthropic_key = os.environ.get('ANTHROPIC_API_KEY', '')
_openai_key    = os.environ.get('OPENAI_API_KEY', '')
assert _anthropic_key, 'ANTHROPIC_API_KEY not set — add it to .env'
assert _openai_key,    'OPENAI_API_KEY not set — add it to .env'

GEN_MODEL     = 'claude-haiku-4-5-20251001'  # Generation and regeneration
EMBED_MODEL   = 'text-embedding-3-small'
MAX_SENTENCES = 5    # Hard ceiling on output length per run
K_RETRIEVE    = 3    # Documents retrieved per uncertainty trigger
CHUNK_SIZE    = 400
CHUNK_OVERLAP = 60

# Hedging words that signal model uncertainty about a specific fact or figure.
# When any of these appear in a generated sentence, retrieval is triggered.
# In full FLARE these are replaced by token log-probabilities (unavailable here).
HEDGE_WORDS = [
    'unclear', 'may', 'possibly', 'approximately', 'around',
    'uncertain', 'roughly', 'might', 'perhaps', 'estimate',
    'likely', 'probably',
]

client        = Anthropic()
lc_embeddings = OpenAIEmbeddings(model=EMBED_MODEL)

print('Setup complete')
print(f'  Generation model : {GEN_MODEL}')
print(f'  Max sentences    : {MAX_SENTENCES}')
print(f'  K retrieve       : {K_RETRIEVE} docs per trigger')
print(f'  Hedge words      : {HEDGE_WORDS}')
print()
print('NOTE: This is a simplified FLARE implementation.')
print('Full FLARE uses per-token log-probabilities to detect uncertainty.')
print('This notebook uses hedging language detection as a practical substitute.')

## Cell 2: Data — Basel III Excerpt
**What this demonstrates**: Loading the Basel III regulatory text as the knowledge base. The document contains precise regulatory figures (capital ratios, buffers, phase-in dates) that a model may hedge about when generating from parametric knowledge — making it a natural corpus for FLARE. The hedging/retrieval trigger fires most naturally on sentences about specific thresholds or timelines.
**Expected output**: Chunk count and a preview of the types of sentences likely to trigger retrieval.

In [ ]:
_base_candidates = [
    pathlib.Path('../../shared/sample_data'),
    pathlib.Path('shared/sample_data'),
]
BASE_DIR = next((p for p in _base_candidates if p.exists()), None)
assert BASE_DIR is not None, 'Cannot find shared/sample_data'

raw_text = (BASE_DIR / 'basel_iii_excerpt.txt').read_text(encoding='utf-8')

splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=['\n\n', '\n', '.', ' '],
)
chunks = [d.page_content for d in splitter.create_documents([raw_text])]
lc_docs = [Document(page_content=c, metadata={'source': 'basel_iii_excerpt'}) for c in chunks]

print(f'Loaded: {len(raw_text):,} characters → {len(chunks)} chunks')

print('\nBuilding Chroma index...', end=' ', flush=True)
vectorstore = Chroma.from_documents(
    lc_docs, lc_embeddings, collection_name='flare_basel_iii'
)
print('done')

print()
print('Sentence types likely to trigger retrieval (contain hedge words):')
print('  CONFIDENT  → "Basel III establishes a global regulatory framework"')
print('  HEDGED     → "The minimum CET1 ratio may be approximately 4.5%"')
print('  CONFIDENT  → "Banks must hold sufficient capital to absorb losses"')
print('  HEDGED     → "The leverage ratio is possibly around 3%"')
print()
print('FLARE: hedged sentences → retrieve → regenerate with grounded figures.')
print('Confident sentences pass through without retrieval.')

## Cell 3: Core — Simplified FLARE Loop
**What this demonstrates**: Three functions implement the FLARE loop. `detect_hedges` scans a sentence for hedging words (the uncertainty proxy). `generate_next_sentence` generates one sentence given the query, prior output, and any injected context. `flare_generate` runs the loop: generate → detect → retrieve if hedged → regenerate with context → accumulate → repeat up to MAX_SENTENCES. A `SentenceResult` dataclass records every decision for observability.
**Expected output**: Pipeline ready confirmation and component descriptions.

In [ ]:
# ── Data type for per-sentence observability ──────────────────────────────────

@dataclass
class SentenceResult:
    """Complete record of one generation step — decisions, retrieval, timing."""
    index             : int
    tentative         : str          # Sentence as first generated (may contain hedges)
    sentence          : str          # Final accepted sentence (after possible regeneration)
    hedges_found      : list         # Hedge words detected in tentative
    retrieval_triggered: bool        # True if retrieval fired on this sentence
    docs_retrieved    : list         # Documents used for regeneration (if any)
    latency_ms        : float


# ── Hedge detector ────────────────────────────────────────────────────────────

def detect_hedges(sentence: str) -> list:
    """Return any hedge words present in the sentence.

    In full FLARE, low-probability tokens serve this role. Here, hedging
    language is used as a proxy — it correlates with model uncertainty
    on precise figures and domain-specific thresholds.

    Args:
        sentence: One generated sentence to inspect.

    Returns:
        list[str] of matched hedge words (empty if sentence is confident).
    """
    lower = sentence.lower()
    return [
        word for word in HEDGE_WORDS
        # Whole-word match — avoids matching 'likely' inside 'unlikely'
        if re.search(r'\b' + re.escape(word) + r'\b', lower)
    ]


# ── Single-sentence generator ─────────────────────────────────────────────────

def generate_next_sentence(
    query      : str,
    prior_text : str,
    context    : str = '',
) -> str:
    """Generate exactly one sentence continuing the answer to query.

    When context is provided (post-retrieval), the model is instructed to
    use those documents and avoid hedging language — the sentence should
    cite specific figures from the retrieved text.

    Args:
        query:      The original question driving generation.
        prior_text: Sentences generated so far (empty on first call).
        context:    Retrieved document text injected after a hedge trigger.

    Returns:
        One sentence as a string.
    """
    parts = [f'Question: {query}']
    if prior_text:
        parts.append(f'Answer so far:\n{prior_text}')
    if context:
        parts.append(
            f'Reference documents (use these to give precise figures):\n{context}\n\n'
            'Using the reference documents above, write the next sentence '
            'with specific, grounded figures. Do not hedge.'
        )
    else:
        parts.append('Write exactly one next sentence of the answer. Stop after the full stop.')

    response = client.messages.create(
        model=GEN_MODEL,
        max_tokens=120,
        system='You are writing a factual explanation. Write exactly one sentence. Stop immediately after the first full stop.',
        messages=[{'role': 'user', 'content': '\n\n'.join(parts)}],
    )
    raw = response.content[0].text.strip()
    # Extract only the first sentence in case the model generates more
    match = re.match(r'([^.!?]+[.!?])', raw)
    return match.group(1).strip() if match else raw.split('\n')[0].strip()


# ── FLARE generation loop ─────────────────────────────────────────────────────

def flare_generate(
    query        : str,
    vectorstore  : Chroma,
    max_sentences: int = MAX_SENTENCES,
    k            : int = K_RETRIEVE,
) -> dict:
    """Simplified FLARE: generate sentence by sentence, retrieve on hedge detection.

    Loop per sentence:
      1. Generate tentative sentence (no docs).
      2. Detect hedge words — uncertainty proxy.
      3. If hedged: retrieve on tentative sentence → inject docs → regenerate.
      4. Accept final sentence and accumulate.
      5. Stop at max_sentences.

    Args:
        query:         The question to answer iteratively.
        vectorstore:   Chroma index over the knowledge base.
        max_sentences: Hard ceiling on total sentences generated.
        k:             Documents to retrieve per trigger.

    Returns:
        dict with keys: query, output, sentences, retrieval_count,
                        total_sentences, total_latency_ms.
    """
    sentence_log: list[SentenceResult] = []
    output_parts: list[str] = []
    retrieval_count = 0
    # Accumulated retrieved context — bounded to avoid context overflow
    injected_context = ''
    t_total = time.perf_counter()

    for i in range(1, max_sentences + 1):
        t_sent = time.perf_counter()
        prior  = ' '.join(output_parts)

        # Step 1: Tentative generation — no retrieved context initially
        tentative = generate_next_sentence(query, prior)

        # Step 2: Hedge detection — the uncertainty proxy
        hedges = detect_hedges(tentative)

        if hedges:
            # Step 3: Retrieve on the hedged sentence as the query
            # The uncertain span is used verbatim — it describes what the model doesn't know
            docs = vectorstore.similarity_search(tentative, k=k)
            retrieval_count += 1
            # Build context string; keep bounded to avoid context overflow
            doc_text = '\n\n'.join(d.page_content for d in docs)
            injected_context = doc_text[:800]   # Bounded window

            # Step 4: Regenerate with retrieved context
            final = generate_next_sentence(query, prior, injected_context)
        else:
            docs  = []
            final = tentative

        output_parts.append(final)
        sentence_log.append(SentenceResult(
            index              = i,
            tentative          = tentative,
            sentence           = final,
            hedges_found       = hedges,
            retrieval_triggered= bool(hedges),
            docs_retrieved     = docs,
            latency_ms         = (time.perf_counter() - t_sent) * 1000,
        ))

    return {
        'query'            : query,
        'output'           : ' '.join(output_parts),
        'sentences'        : sentence_log,
        'retrieval_count'  : retrieval_count,
        'total_sentences'  : len(sentence_log),
        'total_latency_ms' : (time.perf_counter() - t_total) * 1000,
    }


print('Pipeline ready: flare_generate(query, vectorstore)')
print()
print('Components:')
print(f'  detect_hedges         — scans for {len(HEDGE_WORDS)} hedge words (uncertainty proxy)')
print(f'  generate_next_sentence — {GEN_MODEL}, one sentence at a time')
print(f'  flare_generate loop   — max {MAX_SENTENCES} sentences, retrieve on hedge')

## Cell 4: Run — Long-Form Basel III Explanation
**What this demonstrates**: A long-form query that naturally produces a mix of confident and hedged sentences. Sentences about general principles are generated without retrieval. Sentences about specific ratios, buffers, or phase-in timelines will likely include hedging language and trigger retrieval — producing grounded figures from the Basel III document.
**Expected output**: Sentence-by-sentence generation log showing which sentences triggered retrieval, followed by the final grounded output.

In [ ]:
QUERY = 'Explain the full Basel III framework including capital requirements, liquidity standards, and leverage ratios.'

print(f'Query: {QUERY}')
print(f'Max sentences: {MAX_SENTENCES}  |  Hedge words trigger retrieval')
print()

result = flare_generate(QUERY, vectorstore)

# Per-sentence generation log
print('GENERATION LOG')
print('-' * 65)
for sr in result['sentences']:
    icon = chr(9888) if sr.retrieval_triggered else chr(10003)   # ⚠ or ✓
    status = f'RETRIEVED ({", ".join(sr.hedges_found)})' if sr.retrieval_triggered else 'accepted'
    print(f'  Sent {sr.index}  {icon}  {status}')
    if sr.retrieval_triggered:
        print(f'    Tentative: {sr.tentative[:100]}')
        print(f'    Final    : {sr.sentence[:100]}')
    else:
        print(f'    {sr.sentence[:110]}')
    print()

print(f'Retrieval triggered: {result["retrieval_count"]}/{result["total_sentences"]} sentences')
print(f'Total latency      : {result["total_latency_ms"]:.0f} ms')
print()
print('FINAL OUTPUT')
print('=' * 65)
print(result['output'])

## Cell 5: Inspect — Retrieval Triggers, Context Injection, Grounding
**What this demonstrates**: Where FLARE intervened — which hedge words triggered retrieval, which documents were surfaced, and how the tentative sentence changed after grounding. For confident sentences, the before/after is identical. For hedged sentences, the regenerated version should contain specific figures from the retrieved documents.
**Expected output**: Per-sentence breakdown (tentative vs. final), retrieved document previews for triggered sentences, and a summary table.

In [ ]:
# ── Detailed inspection of each sentence ─────────────────────────────────────
for sr in result['sentences']:
    print(f'SENTENCE {sr.index}  |  {sr.latency_ms:.0f} ms')
    print('=' * 65)

    if sr.retrieval_triggered:
        print(f'Hedge words found: {sr.hedges_found}')
        print(f'Retrieval triggered on: "{sr.tentative[:80]}..."')
        print()
        print('Retrieved documents:')
        for j, doc in enumerate(sr.docs_retrieved):
            src = doc.metadata.get('source', 'unknown')
            print(f'  [{j+1}] [{src}] {doc.page_content[:180].replace(chr(10), " ")}...')
        print()
        print(f'Tentative : {sr.tentative}')
        print(f'Regenerated: {sr.sentence}')
        # Check if specific numbers were added post-retrieval
        numbers_in_final = re.findall(r'\d+(?:\.\d+)?%?', sr.sentence)
        numbers_in_tent  = re.findall(r'\d+(?:\.\d+)?%?', sr.tentative)
        new_numbers = [n for n in numbers_in_final if n not in numbers_in_tent]
        if new_numbers:
            print(f'Figures added by retrieval: {new_numbers}')
    else:
        print('No hedges detected — accepted on first pass.')
        print(f'Sentence: {sr.sentence}')

    print()

# ── Summary table ─────────────────────────────────────────────────────────────
print('SUMMARY TABLE')
print('=' * 65)
print(f'{"Sent":5s}  {"Hedges":20s}  {"Retrieved":9s}  {"ms":>6}')
print(f'{"-"*5}  {"-"*20}  {"-"*9}  {"-"*6}')
for sr in result['sentences']:
    hedge_str = ', '.join(sr.hedges_found[:3]) if sr.hedges_found else '—'
    ret_str   = f'Yes ({len(sr.docs_retrieved)} docs)' if sr.retrieval_triggered else 'No'
    print(f'{sr.index:5d}  {hedge_str:20s}  {ret_str:9s}  {sr.latency_ms:6.0f}')

print()
print(f'Sentences with retrieval : {result["retrieval_count"]}')
print(f'Sentences without        : {result["total_sentences"] - result["retrieval_count"]}')
print()
print('Key FLARE property: retrieval fires only where hedges appear.')
print('Confident sentences pass through without any retrieval cost.')

## Cell 6: Fintech — Iterative Compliance Report Generation
**What this demonstrates**: FLARE applied to a compliance report writing task — the primary production use case for this pattern. The model writes a risk report sentence by sentence. Sentences about general obligations are generated from parametric knowledge; sentences about specific capital ratios, buffer requirements, or threshold figures trigger retrieval and are grounded in the Basel III document. The generation log shows the difference between a report written purely from parametric knowledge and one that retrieves on demand.
**Expected output**: Generation log for the compliance query, final grounded report, and a comparison of retrieval-triggered vs. confident sentence counts.

In [ ]:
REPORT_QUERY = (
    'Write a compliance report assessing a commercial bank\'s '
    'Basel III Tier 1 capital adequacy obligations, '
    'including minimum ratio requirements and conservation buffers.'
)

print('FINTECH: ITERATIVE COMPLIANCE REPORT GENERATION')
print('Use case: compliance officer drafting a Basel III capital adequacy assessment')
print()
print(f'Query: {REPORT_QUERY}')
print()

report_result = flare_generate(REPORT_QUERY, vectorstore)

# Generation trace
print('GENERATION TRACE')
print('-' * 65)
for sr in report_result['sentences']:
    if sr.retrieval_triggered:
        print(f'  [{sr.index}] HEDGED  → triggered retrieval on: {sr.hedges_found}')
        print(f'       Before: {sr.tentative[:90]}')
        print(f'       After : {sr.sentence[:90]}')
    else:
        print(f'  [{sr.index}] CONFIDENT → {sr.sentence[:90]}')
    print()

# Final report
print('FINAL COMPLIANCE REPORT')
print('=' * 65)
print(report_result['output'])
print()

# Cost comparison: FLARE vs naive full-context RAG
total_s    = report_result['total_sentences']
retrieved  = report_result['retrieval_count']
confident  = total_s - retrieved

print('RETRIEVAL EFFICIENCY')
print('=' * 65)
print(f'  Total sentences        : {total_s}')
print(f'  Confident (no retrieval): {confident}  — generated from parametric knowledge')
print(f'  Hedged (retrieval fired): {retrieved}  — grounded in Basel III document')
if total_s > 0:
    pct = 100 * confident // total_s
    print(f'  Retrieval rate         : {retrieved}/{total_s} sentences ({100-pct}%)')
    print(f'  Upfront-retrieval equivalent: would load all docs for all {total_s} sentences')
    print(f'  FLARE: retrieved only for {retrieved}/{total_s} sentences where needed')
print()
print(f'Total latency: {report_result["total_latency_ms"]:.0f} ms')
print()
print('NOTE: This simplified FLARE uses hedging language as an uncertainty proxy.')
print('In the original paper, per-token log-probabilities replace hedge detection.')
print('The pattern — generate, detect uncertainty, retrieve, regenerate — is identical.')